# Bloque 5 — Visualización con Plotly y Faker
## Proyecto: Red Social Ficticia — Gráficos y datos sintéticos

En este bloque **representamos los datos con gráficos interactivos** y **ampliamos el dataset con datos sintéticos**.

**Parte A — Visualización con Plotly:** gráficos interactivos (barras, dispersión, líneas, histogramas, cajas, violín, mapa de calor), paneles con varios gráficos (`subplots`) y exportación a **HTML** y **PNG**. Los gráficos parten del fichero `../data/dataset_procesado.parquet` que generaste en el **Bloque 4**.

**Parte B — Faker:** generación de datos ficticios realistas y **reproducibles**, proveedores personalizados y ampliación del dataset de 100 a más de 5000 usuarios manteniendo su esquema.

> Requisitos: `plotly` (con `kaleido` para exportar a PNG) y `faker`. Antes de empezar, comprueba que existe `../data/dataset_procesado.parquet` (ejercicio 8 del Bloque 4).

---
### Objetivos
- Crear gráficos interactivos con Plotly Express: barras, dispersión, líneas, histogramas, cajas, violín y mapas de calor.
- Personalizar títulos, ejes, colores, transparencias, tamaños, información al pasar el ratón y plantilla visual.
- Combinar varios gráficos en un panel (`make_subplots`) con `graph_objects`.
- Exportar figuras a HTML (interactivas) y a PNG (estáticas).
- Generar datos ficticios con Faker de forma reproducible y con proveedores propios.
- Ampliar un dataset manteniendo columnas, tipos de dato, identificadores y distribuciones, y guardarlo en CSV y Parquet.

### Cómo trabajar este cuaderno
1. Ejecuta primero la celda de carga de datos.
2. Completa los **ejercicios básicos** en orden: cada celda indica el nombre de las variables que debes crear. Sustituye cada `None` (o `pass`) por tu código. Los ejercicios se apoyan unos en otros.
3. Los enunciados explican **qué** se pide y describen el resultado esperado con ejemplos inventados, pero no indican qué métodos usar: búscalos en tus apuntes y en las listas de ejercicios.
4. Continúa con los **ejercicios avanzados** y, para terminar, implementa las funciones de la sección "Entrega" en `../scripts/`.

## 1. Carga de datos

In [ ]:
# Importamos las librerías del bloque y cargamos los datos: el fichero Parquet
# generado en el Bloque 4 (publicaciones combinadas con sus autores y su
# 'engagement_rate') y la tabla de usuarios. Esta celda ya está resuelta, no
# la modifiques.
import os
import random
from datetime import date

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from faker import Faker
from plotly.subplots import make_subplots

RUTA_PARQUET = '../data/dataset_procesado.parquet'
if not os.path.exists(RUTA_PARQUET):
    raise FileNotFoundError('Falta ../data/dataset_procesado.parquet: completa antes el ejercicio 8 del Bloque 4.')

df_procesado = pd.read_parquet(RUTA_PARQUET)

# Las columnas de publicación del Parquet forman la tabla de publicaciones
# (leemos solo esas columnas).
COLUMNAS_PUBLICACION = ['id_publicacion', 'id_usuario', 'fecha', 'tipo_contenido', 'likes',
                        'comentarios', 'compartidos', 'visualizaciones', 'hashtags']
df_publicaciones = pd.read_parquet(RUTA_PARQUET, columns=COLUMNAS_PUBLICACION)

df_usuarios = pd.read_csv('../data/usuarios.csv')
df_usuarios['edad'] = df_usuarios['edad'].fillna(df_usuarios['edad'].median())
df_usuarios['ciudad'] = df_usuarios['ciudad'].fillna('Desconocido')
df_usuarios['bio'] = df_usuarios['bio'].fillna('Desconocido')

os.makedirs('../output_graficos', exist_ok=True)
print('df_procesado:', df_procesado.shape)
print('df_publicaciones:', df_publicaciones.shape)
print('df_usuarios:', df_usuarios.shape)


---
## Ejercicios básicos

### Parte A — Visualización con Plotly

Todos los gráficos deben tener título y etiquetas de ejes descriptivos en español. Los ejemplos de cada enunciado describen gráficos inventados y **nunca** son la solución. Las figuras se guardan en la carpeta `../output_graficos/`, que ya está creada. Salvo que se indique lo contrario, trabaja con `df_procesado`.

In [ ]:
# ─── Ejercicio 1 — Plotly Express: gráfico de barras ────────────────────────
# 1. Guarda en 'publicaciones_por_ciudad' una tabla con las columnas 'ciudad'
#    y 'publicaciones' con el número de publicaciones de cada ciudad de
#    'df_procesado', quedándote con las 8 ciudades con más publicaciones,
#    de mayor a menor.
# 2. Guarda en 'fig_barras' un gráfico de barras interactivo de Plotly con
#    esas 8 ciudades: la ciudad en el eje horizontal y las publicaciones en
#    el vertical. Requisitos:
#      - cada barra de un color distinto según su ciudad
#      - el valor de cada barra escrito sobre ella
#      - título y etiquetas de los dos ejes en español
#      - plantilla visual 'plotly_white' y sin leyenda (los nombres ya
#        aparecen en el eje horizontal)
#    Por ejemplo, con 3 ciudades (Madrid 4, Bilbao 3 y Sevilla 1) habría 3
#    barras de colores distintos, de alturas 4, 3 y 1, con esos números
#    encima (esto NO es la solución, tu gráfico tiene 8 barras).
# 3. Exporta la figura a '../output_graficos/barras_ciudades.html' y como
#    imagen PNG a '../output_graficos/barras_ciudades.png'.

publicaciones_por_ciudad = df_procesado['ciudad'].value_counts().head(8).reset_index()
publicaciones_por_ciudad.columns = ['ciudad', 'publicaciones']

fig_barras = px.bar(
    publicaciones_por_ciudad,
    x='ciudad',
    y='publicaciones',
    color='ciudad',
    text='publicaciones',
    title='Publicaciones por ciudad (top 8)',
    labels={'ciudad': 'Ciudad', 'publicaciones': 'Número de publicaciones'},
    template='plotly_white',
)
fig_barras.update_layout(showlegend=False)

fig_barras.write_html('../output_graficos/barras_ciudades.html')
fig_barras.write_image('../output_graficos/barras_ciudades.png')
fig_barras.show()


In [ ]:
# ─── Ejercicio 2 — Plotly Express: diagrama de dispersión ───────────────────
# Guarda en 'fig_dispersion' un diagrama de dispersión interactivo de las
# publicaciones de 'df_procesado': los 'likes' en el eje horizontal y las
# 'visualizaciones' en el vertical. Requisitos:
#   - cada punto coloreado según el 'tipo_contenido' de la publicación
#   - el tamaño de cada punto proporcional a sus 'comentarios'
#   - transparencia 0.7 para ver los puntos que se solapan
#   - al pasar el ratón por un punto debe aparecer el nombre y la ciudad del
#     autor de la publicación
#   - cada punto con un borde gris oscuro de 1 píxel de grosor
#   - título y etiquetas de ejes en español
# Por ejemplo, una publicación de tipo 'video' con 2000 likes, 15000
# visualizaciones y 300 comentarios sería un punto grande de un color
# propio de 'video' en la posición (2000, 15000); al pasar el ratón por
# encima aparecería el autor (esto NO es la solución).
#
# Exporta la figura a '../output_graficos/dispersion_likes.html' y como
# imagen PNG a '../output_graficos/dispersion_likes.png'.

fig_dispersion = px.scatter(
    df_procesado,
    x='likes',
    y='visualizaciones',
    color='tipo_contenido',
    size='comentarios',
    opacity=0.7,
    hover_data=['nombre', 'ciudad'],
    title='Likes frente a visualizaciones',
    labels={'likes': 'Likes', 'visualizaciones': 'Visualizaciones', 'tipo_contenido': 'Tipo de contenido'},
)
fig_dispersion.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))

fig_dispersion.write_html('../output_graficos/dispersion_likes.html')
fig_dispersion.write_image('../output_graficos/dispersion_likes.png')
fig_dispersion.show()


In [ ]:
# ─── Ejercicio 3 — Plotly Express: gráficos de líneas ───────────────────────
# 1. Guarda en 'publicaciones_por_mes' una tabla con las columnas 'mes'
#    (formato 'AAAA-MM', calculado a partir de la columna 'fecha', que tiene
#    el formato 'AAAA-MM-DD') y 'publicaciones' (número de publicaciones de
#    ese mes), ordenada cronológicamente.
#    Por ejemplo, para las fechas 2023-01-05, 2023-01-20 y 2023-02-03 la
#    tabla tendría dos filas: (2023-01, 2) y (2023-02, 1) (esto NO es la
#    solución).
# 2. Guarda en 'fig_lineas' un gráfico de líneas con las publicaciones por
#    mes, con un marcador en cada punto, título y etiquetas de ejes.
# 3. Guarda en 'publicaciones_mes_tipo' una tabla con las columnas 'mes',
#    'tipo_contenido' y 'publicaciones' (publicaciones de cada tipo de
#    contenido en cada mes) y en 'fig_lineas_tipo' un gráfico de líneas con
#    una línea de distinto color para cada tipo de contenido.
# 4. Exporta las dos figuras a HTML en '../output_graficos/' con los nombres
#    'lineas_publicaciones.html' y 'lineas_por_tipo.html'.

df_con_mes = df_procesado.assign(mes=df_procesado['fecha'].str[:7])

publicaciones_por_mes = (
    df_con_mes.groupby('mes').size().reset_index(name='publicaciones').sort_values('mes')
)
fig_lineas = px.line(
    publicaciones_por_mes,
    x='mes',
    y='publicaciones',
    markers=True,
    title='Publicaciones por mes',
    labels={'mes': 'Mes', 'publicaciones': 'Número de publicaciones'},
)

publicaciones_mes_tipo = (
    df_con_mes.groupby(['mes', 'tipo_contenido']).size().reset_index(name='publicaciones').sort_values('mes')
)
fig_lineas_tipo = px.line(
    publicaciones_mes_tipo,
    x='mes',
    y='publicaciones',
    color='tipo_contenido',
    title='Publicaciones por mes y tipo de contenido',
    labels={'mes': 'Mes', 'publicaciones': 'Número de publicaciones', 'tipo_contenido': 'Tipo de contenido'},
)

fig_lineas.write_html('../output_graficos/lineas_publicaciones.html')
fig_lineas_tipo.write_html('../output_graficos/lineas_por_tipo.html')
print(publicaciones_por_mes.head(3))
fig_lineas.show()


In [ ]:
# ─── Ejercicio 4 — Plotly Express: distribuciones ───────────────────────────
# 1. Guarda en 'fig_histograma' un histograma del 'engagement_rate' de
#    'df_procesado' con 30 tramos, con un color distinto para cada
#    'tipo_cuenta', las barras de los distintos grupos superpuestas con
#    transparencia 0.6 y, encima del histograma, un diagrama de cajas
#    marginal.
# 2. Guarda en 'fig_cajas' un diagrama de cajas de los 'likes' para cada
#    'tipo_contenido', mostrando además todos los puntos.
#    Por ejemplo, habría una caja por tipo de contenido, con su mediana, sus
#    cuartiles y los puntos de todas sus publicaciones (esto NO es la
#    solución).
# 3. Guarda en 'fig_violin' un diagrama de violín del 'engagement_rate' por
#    'tipo_cuenta', con un diagrama de cajas dentro de cada violín.
# 4. Guarda en 'mediana_engagement_por_cuenta' la mediana del
#    'engagement_rate' de cada 'tipo_cuenta', para comprobar
#    numéricamente lo que muestran los gráficos.
# 5. Exporta las tres figuras a HTML en '../output_graficos/' con los
#    nombres 'histograma_engagement.html', 'cajas_likes.html' y
#    'violin_engagement.html'.

fig_histograma = px.histogram(
    df_procesado,
    x='engagement_rate',
    nbins=30,
    color='tipo_cuenta',
    barmode='overlay',
    opacity=0.6,
    marginal='box',
    title='Distribución del engagement por tipo de cuenta',
    labels={'engagement_rate': 'Engagement', 'tipo_cuenta': 'Tipo de cuenta'},
)
fig_cajas = px.box(
    df_procesado,
    x='tipo_contenido',
    y='likes',
    points='all',
    title='Likes por tipo de contenido',
    labels={'tipo_contenido': 'Tipo de contenido', 'likes': 'Likes'},
)
fig_violin = px.violin(
    df_procesado,
    x='tipo_cuenta',
    y='engagement_rate',
    box=True,
    title='Engagement por tipo de cuenta',
    labels={'tipo_cuenta': 'Tipo de cuenta', 'engagement_rate': 'Engagement'},
)
mediana_engagement_por_cuenta = df_procesado.groupby('tipo_cuenta')['engagement_rate'].median()

fig_histograma.write_html('../output_graficos/histograma_engagement.html')
fig_cajas.write_html('../output_graficos/cajas_likes.html')
fig_violin.write_html('../output_graficos/violin_engagement.html')
print(mediana_engagement_por_cuenta.round(3))
fig_histograma.show()


In [ ]:
# ─── Ejercicio 5 — Plotly Express: mapa de calor de correlaciones ───────────
# Un mapa de calor colorea cada celda de una matriz según su valor; es
# habitual para matrices de correlación.
#
# 1. Guarda en 'matriz_correlacion' la matriz de correlación entre las
#    columnas 'likes', 'comentarios', 'compartidos', 'visualizaciones' y
#    'engagement_rate' de 'df_procesado' (5 filas y 5 columnas).
# 2. Guarda en 'fig_mapa_calor' un mapa de calor de esa matriz con:
#      - el valor de cada celda escrito con 2 decimales
#      - escala de color divergente de rojo a azul, con el 0 en el centro
#        (rango de -1 a 1 para poder comparar figuras)
#      - título descriptivo
#    Por ejemplo, la diagonal principal siempre valdría 1 (cada variable
#    está perfectamente correlacionada consigo misma).
# 3. Guarda en 'variable_mas_correlada_con_likes' el nombre de la variable,
#    distinta de 'likes', que tiene mayor correlación con 'likes'.
# 4. Exporta la figura a '../output_graficos/mapa_calor_correlaciones.html'.

columnas_numericas = ['likes', 'comentarios', 'compartidos', 'visualizaciones', 'engagement_rate']
matriz_correlacion = df_procesado[columnas_numericas].corr()

fig_mapa_calor = px.imshow(
    matriz_correlacion,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlación entre las métricas de las publicaciones',
)
variable_mas_correlada_con_likes = matriz_correlacion['likes'].drop('likes').idxmax()

fig_mapa_calor.write_html('../output_graficos/mapa_calor_correlaciones.html')
print(matriz_correlacion.shape, variable_mas_correlada_con_likes)
fig_mapa_calor.show()


In [ ]:
# ─── Ejercicio 6 — Plotly: panel con varios gráficos ────────────────────────
# Guarda en 'fig_dashboard' UNA figura de Plotly, con Graph Objects, con una
# cuadrícula de 2 filas x 2 columnas que contenga:
#   [fila 1, columna 1] Barras: 'publicaciones_por_ciudad' (ejercicio 1).
#   [fila 1, columna 2] Sectores: número de publicaciones de cada
#                       'tipo_cuenta' de 'df_procesado'.
#   [fila 2, columna 1] Líneas con marcadores: 'publicaciones_por_mes'
#                       (ejercicio 3).
#   [fila 2, columna 2] Histograma: 'likes' de 'df_procesado'.
# Requisitos: cada subgráfico con su propio título, figura de 800 píxeles de
# alto, título general 'Dashboard Red Social Ficticia', plantilla visual
# 'plotly_white' y sin leyenda. Ojo: los gráficos de sectores necesitan un
# tipo de subgráfico propio.
#
# Exporta la figura a '../output_graficos/dashboard.html' y como imagen PNG a
# '../output_graficos/dashboard.png'.

publicaciones_por_cuenta = df_procesado['tipo_cuenta'].value_counts()

fig_dashboard = make_subplots(
    rows=2,
    cols=2,
    specs=[[{}, {'type': 'domain'}], [{}, {}]],
    subplot_titles=[
        'Publicaciones por ciudad',
        'Publicaciones por tipo de cuenta',
        'Publicaciones por mes',
        'Distribución de likes',
    ],
)
fig_dashboard.add_trace(
    go.Bar(x=publicaciones_por_ciudad['ciudad'], y=publicaciones_por_ciudad['publicaciones']),
    row=1, col=1,
)
fig_dashboard.add_trace(
    go.Pie(labels=publicaciones_por_cuenta.index, values=publicaciones_por_cuenta.values),
    row=1, col=2,
)
fig_dashboard.add_trace(
    go.Scatter(x=publicaciones_por_mes['mes'], y=publicaciones_por_mes['publicaciones'], mode='lines+markers'),
    row=2, col=1,
)
fig_dashboard.add_trace(go.Histogram(x=df_procesado['likes']), row=2, col=2)
fig_dashboard.update_layout(
    height=800,
    title_text='Dashboard Red Social Ficticia',
    template='plotly_white',
    showlegend=False,
)

fig_dashboard.write_html('../output_graficos/dashboard.html')
fig_dashboard.write_image('../output_graficos/dashboard.png')
fig_dashboard.show()


### Parte B — Faker

Faker genera datos ficticios que imitan datos reales (nombres, direcciones, correos, fechas...). Es útil para probar aplicaciones y para **ampliar un dataset** sin usar datos reales. Puede trabajar en varios idiomas y su generador se puede **inicializar con una semilla** para que produzca siempre los mismos datos.

In [ ]:
# ─── Ejercicio 7 — Faker: datos reproducibles y proveedor propio ──────────
# 1. Crea un generador de Faker con datos de España (en español) en 'fake' y
#    fija la semilla 2024, de modo que cada ejecución produzca los mismos
#    datos.
# 2. Guarda en 'df_faker_demo' una tabla de 10 usuarios ficticios con las
#    columnas 'nombre', 'apellidos' (dos apellidos), 'email', 'ciudad',
#    'telefono' y 'fecha_registro' (una fecha entre el 2018-01-01 y el
#    2024-06-01, escrita como texto 'AAAA-MM-DD').
#    Por ejemplo, una fila podría ser (esto NO es la solución):
#      nombre  apellidos       email            ciudad   telefono   fecha_registro
#      Lucía   Ortega Ramos    lucia@ejemplo.es Sevilla  600123456  2021-03-14
# 3. Comprueba la reproducibilidad: guarda en 'es_reproducible' un valor
#    True/False que indique si, fijando la semilla 2024 antes de cada una,
#    dos generaciones de 10 nombres de pila dan exactamente la misma lista.
# 4. Guarda en 'nombres_unicos' una lista de 10 nombres de pila SIN
#    repetidos (Faker tiene un modo para garantizar valores únicos).
# 5. Crea un proveedor de datos propio, la clase 'ProveedorRedSocial', con un
#    método 'tipo_cuenta' que devuelva 'free', 'premium' o 'creator' en
#    proporción 6 : 3 : 1, y regístralo en 'fake'. Guarda en
#    'tipos_cuenta_faker' una lista de 10 tipos de cuenta generados con él.

from faker.providers import BaseProvider

fake = Faker('es_ES')
Faker.seed(2024)

df_faker_demo = pd.DataFrame({
    'nombre': [fake.first_name() for _ in range(10)],
    'apellidos': [f'{fake.last_name()} {fake.last_name()}' for _ in range(10)],
    'email': [fake.email() for _ in range(10)],
    'ciudad': [fake.city() for _ in range(10)],
    'telefono': [fake.phone_number() for _ in range(10)],
    'fecha_registro': [
        fake.date_between(date(2018, 1, 1), date(2024, 6, 1)).isoformat() for _ in range(10)
    ],
})

Faker.seed(2024)
primera_generacion = [fake.first_name() for _ in range(10)]
Faker.seed(2024)
segunda_generacion = [fake.first_name() for _ in range(10)]
es_reproducible = primera_generacion == segunda_generacion

nombres_unicos = [fake.unique.first_name() for _ in range(10)]


class ProveedorRedSocial(BaseProvider):
    def tipo_cuenta(self):
        return self.random_element(['free'] * 6 + ['premium'] * 3 + ['creator'])


fake.add_provider(ProveedorRedSocial)
tipos_cuenta_faker = [fake.tipo_cuenta() for _ in range(10)]
print(df_faker_demo.head(3))
print(es_reproducible, len(set(nombres_unicos)), tipos_cuenta_faker)


In [ ]:
# ─── Ejercicio 8 — Faker: ampliar el dataset de usuarios ────────────────────
# Vas a pasar de 100 a 5000 usuarios. Los usuarios nuevos deben ser
# coherentes con los reales: mismas columnas, mismos tipos y valores
# plausibles.
#
# 1. Fija la semilla 2024 en Faker y en el módulo 'random' de Python (para
#    que el resultado sea reproducible) y guarda en 'df_nuevos' 4900
#    usuarios nuevos con exactamente las mismas columnas, en el mismo orden
#    y con los mismos tipos de dato que 'df_usuarios'. Reglas:
#      - 'id_usuario': continúa desde el 101 (101, 102, 103...)
#      - 'nombre' y 'apellidos' (dos apellidos): generados con Faker
#      - 'edad': entre 18 y 55
#      - 'ciudad', 'pais' y 'genero': elegidos al azar entre los valores que
#        ya existen en 'df_usuarios'
#      - 'fecha_registro': entre 2018-01-01 y 2024-06-01 (texto AAAA-MM-DD)
#      - 'tipo_cuenta': 60% free, 30% premium y 10% creator
#      - 'seguidores': entre 1000 y 50000 si es creator, entre 10 y 5000 si no
#      - 'seguidos': entre 20 y 1000; 'publicaciones_totales': entre 0 y 500
#      - 'activo': verdadero en el 80% de los casos
#      - 'bio': una frase generada con Faker
#    Por ejemplo, un usuario nuevo podría ser (esto NO es la solución):
#      id_usuario 101, nombre Marta, tipo_cuenta free, seguidores 1830, ...
# 2. Guarda en 'df_grande' los usuarios originales seguidos de los nuevos
#    (5000 filas, con índice de 0 a 4999).
# 3. Guarda en 'ids_unicos' si todos los 'id_usuario' de 'df_grande' son
#    distintos, en 'columnas_iguales' si tiene las mismas columnas y orden
#    que 'df_usuarios', y en 'tipos_iguales' si cada columna tiene el mismo
#    tipo de dato que en 'df_usuarios'.
# 4. Guarda en 'proporcion_tipos' el porcentaje de cada 'tipo_cuenta' entre
#    los usuarios NUEVOS, con 1 decimal (deberían parecerse a 60 / 30 / 10).
# 5. Guarda 'df_grande' en '../data/usuarios_grande.csv' y en
#    '../data/usuarios_grande.parquet' (compresión snappy), ambos sin
#    índice, y guarda en 'tamano_csv' y 'tamano_parquet' el tamaño en bytes
#    de cada fichero.
# 6. Lee del fichero Parquet SOLO las columnas 'id_usuario', 'tipo_cuenta' y
#    'seguidores' y guarda la tabla en 'df_lectura_parcial' (5000 filas y 3
#    columnas), como hiciste en el Bloque 4.

Faker.seed(2024)
random.seed(2024)
fake = Faker('es_ES')

ciudades = df_usuarios['ciudad'].unique().tolist()
paises = df_usuarios['pais'].unique().tolist()
generos = df_usuarios['genero'].unique().tolist()

filas = []
for indice in range(4900):
    tipo = random.choices(['free', 'premium', 'creator'], weights=[60, 30, 10])[0]
    filas.append({
        'id_usuario': 101 + indice,
        'nombre': fake.first_name(),
        'apellidos': f'{fake.last_name()} {fake.last_name()}',
        'edad': random.randint(18, 55),
        'ciudad': random.choice(ciudades),
        'pais': random.choice(paises),
        'genero': random.choice(generos),
        'fecha_registro': fake.date_between(date(2018, 1, 1), date(2024, 6, 1)).isoformat(),
        'tipo_cuenta': tipo,
        'seguidores': random.randint(1000, 50000) if tipo == 'creator' else random.randint(10, 5000),
        'seguidos': random.randint(20, 1000),
        'publicaciones_totales': random.randint(0, 500),
        'activo': random.random() < 0.8,
        'bio': fake.catch_phrase(),
    })
df_nuevos = pd.DataFrame(filas, columns=df_usuarios.columns).astype(df_usuarios.dtypes.to_dict())

df_grande = pd.concat([df_usuarios, df_nuevos], ignore_index=True)

ids_unicos = bool(df_grande['id_usuario'].is_unique)
columnas_iguales = list(df_grande.columns) == list(df_usuarios.columns)
tipos_iguales = bool((df_grande.dtypes == df_usuarios.dtypes).all())
proporcion_tipos = (df_nuevos['tipo_cuenta'].value_counts(normalize=True) * 100).round(1)

df_grande.to_csv('../data/usuarios_grande.csv', index=False)
df_grande.to_parquet('../data/usuarios_grande.parquet', compression='snappy', index=False)
tamano_csv = os.path.getsize('../data/usuarios_grande.csv')
tamano_parquet = os.path.getsize('../data/usuarios_grande.parquet')

df_lectura_parcial = pd.read_parquet(
    '../data/usuarios_grande.parquet', columns=['id_usuario', 'tipo_cuenta', 'seguidores']
)
print(len(df_grande), ids_unicos, columnas_iguales, tipos_iguales)
print(proporcion_tipos.to_dict())
print('CSV:', tamano_csv, 'bytes | Parquet:', tamano_parquet, 'bytes |', df_lectura_parcial.shape)


### Entrega — `../scripts/pipeline.py`

Implementa estas funciones en `../scripts/pipeline.py`. Cada una está vinculada a uno de los ejercicios básicos anteriores:

| Función | Descripción |
|---|---|
| `exportar_grafico(fig, ruta_html, ruta_png=None)` | Guarda una figura de Plotly en HTML y, si se indica `ruta_png`, también en PNG; crea las carpetas que falten y devuelve `None` (ejercicios 1 a 6) |
| `publicaciones_por_mes(df)` | Devuelve un DataFrame con las columnas `mes` (formato `AAAA-MM`) y `publicaciones`, ordenado cronológicamente (ejercicio 3) |
| `crear_dashboard(df)` | Devuelve una figura de Plotly con el panel de 2 x 2 gráficos del ejercicio 6 (barras por ciudad, sectores por tipo de cuenta, líneas por mes e histograma de likes), sin modificar el DataFrame (ejercicio 6) |
| `generar_datos_demo(n=10, seed=2024)` | Devuelve un DataFrame con `n` usuarios ficticios de Faker (`nombre`, `apellidos`, `email`, `ciudad`, `telefono`, `fecha_registro`), reproducible con la misma `seed` (ejercicio 7) |
| `ampliar_dataset(df_usuarios, n_nuevos)` | Devuelve los usuarios originales seguidos de `n_nuevos` usuarios ficticios generados con Faker, con ids que continúan desde el máximo, mismas columnas y tipos y las proporciones del ejercicio 8 (ejercicio 8) |

Comprueba tu implementación con:

```bash
python -m pytest ../tests/test_bloque5.py -v
```

---
## Ejercicios avanzados

Estos ejercicios exigen validar los resultados, reutilizar el código en funciones y cuidar la coherencia de los datos sintéticos. Igual que en los básicos, los enunciados no indican qué métodos usar y los ejemplos usan datos inventados: nunca son la solución. Se apoyan en las figuras y tablas de los ejercicios básicos.

In [ ]:
# ─── Ejercicio A.1 — Exportación de figuras por lotes ───────────────────────
# 1. Define una función 'exportar_figuras_avanzadas(figuras, carpeta)' que
#    reciba un diccionario {nombre: figura de Plotly} y una carpeta, guarde
#    cada figura como '<carpeta>/<nombre>.html' (creando la carpeta si no
#    existe) y devuelva la lista con las rutas de los ficheros creados, en
#    el mismo orden que el diccionario.
#    Por ejemplo, para {'barras': fig_a, 'lineas': fig_b} y la carpeta
#    'salida' devolvería ['salida/barras.html', 'salida/lineas.html'] (esto
#    NO es la solución).
# 2. Guarda en 'rutas_exportadas' el resultado de exportar las figuras
#    'fig_barras', 'fig_dispersion' y 'fig_lineas' (con los nombres
#    'barras', 'dispersion' y 'lineas') a la carpeta
#    '../output_graficos/avanzado'.
# 3. Guarda en 'todas_existen' si los tres ficheros existen realmente y en
#    'tamanos_html' una lista con el tamaño en bytes de cada uno.

def exportar_figuras_avanzadas(figuras, carpeta):
    os.makedirs(carpeta, exist_ok=True)
    rutas = []
    for nombre, figura in figuras.items():
        ruta = os.path.join(carpeta, f'{nombre}.html')
        figura.write_html(ruta)
        rutas.append(ruta)
    return rutas


rutas_exportadas = exportar_figuras_avanzadas(
    {'barras': fig_barras, 'dispersion': fig_dispersion, 'lineas': fig_lineas},
    '../output_graficos/avanzado',
)
todas_existen = all(os.path.exists(ruta) for ruta in rutas_exportadas)
tamanos_html = [os.path.getsize(ruta) for ruta in rutas_exportadas]
print(rutas_exportadas, todas_existen, tamanos_html)


In [ ]:
# ─── Ejercicio A.2 — Ampliación reproducible y coherente ────────────────────
# 1. Define una función 'ampliar_dataset_avanzado(df_usuarios, n, seed=42)'
#    que devuelva los usuarios originales (sin cambios y al principio)
#    seguidos de 'n' usuarios ficticios con las mismas reglas del ejercicio 8
#    (ids continuando desde el máximo, mismas columnas y tipos, 60/30/10 de
#    tipos de cuenta, seguidores según el tipo, fechas entre 2018-01-01 y
#    2024-06-01...). Con la MISMA 'seed' debe devolver siempre exactamente
#    los mismos usuarios, y con 'n' igual o menor que 0 debe devolver una
#    copia de la tabla original.
# 2. Guarda en 'df_grande_avanzado' el resultado de ampliar 'df_usuarios' con
#    4900 usuarios y la semilla 42.
# 3. Guarda en 'es_reproducible_avanzado' si llamar dos veces a la función
#    con la misma semilla da tablas idénticas, y en 'semillas_distintas' si
#    con semillas distintas los usuarios nuevos SÍ son distintos.
# 4. Guarda en 'originales_intactos' si las primeras 100 filas de
#    'df_grande_avanzado' son idénticas a 'df_usuarios'.
# 5. Guarda en 'fechas_validas' si todas las fechas de registro de los
#    usuarios nuevos están dentro del periodo permitido.

def ampliar_dataset_avanzado(df_usuarios, n, seed=42):
    if n <= 0:
        return df_usuarios.copy()

    fake = Faker('es_ES')
    fake.seed_instance(seed)
    azar = random.Random(seed)

    id_inicio = int(df_usuarios['id_usuario'].max()) + 1
    ciudades = df_usuarios['ciudad'].dropna().unique().tolist()
    paises = df_usuarios['pais'].dropna().unique().tolist()
    generos = df_usuarios['genero'].dropna().unique().tolist()
    activo_es_bool = pd.api.types.is_bool_dtype(df_usuarios['activo'])

    filas = []
    for indice in range(n):
        tipo = azar.choices(['free', 'premium', 'creator'], weights=[60, 30, 10])[0]
        activo = azar.random() < 0.8
        filas.append({
            'id_usuario': id_inicio + indice,
            'nombre': fake.first_name(),
            'apellidos': f'{fake.last_name()} {fake.last_name()}',
            'edad': azar.randint(18, 55),
            'ciudad': azar.choice(ciudades),
            'pais': azar.choice(paises),
            'genero': azar.choice(generos),
            'fecha_registro': fake.date_between(date(2018, 1, 1), date(2024, 6, 1)).isoformat(),
            'tipo_cuenta': tipo,
            'seguidores': azar.randint(1000, 50000) if tipo == 'creator' else azar.randint(10, 5000),
            'seguidos': azar.randint(20, 1000),
            'publicaciones_totales': azar.randint(0, 500),
            'activo': activo if activo_es_bool else str(activo),
            'bio': fake.catch_phrase(),
        })

    nuevos = pd.DataFrame(filas, columns=df_usuarios.columns).astype(df_usuarios.dtypes.to_dict())
    return pd.concat([df_usuarios, nuevos], ignore_index=True)


df_grande_avanzado = ampliar_dataset_avanzado(df_usuarios, 4900, seed=42)
es_reproducible_avanzado = ampliar_dataset_avanzado(df_usuarios, 200, seed=7).equals(
    ampliar_dataset_avanzado(df_usuarios, 200, seed=7)
)
semillas_distintas = not ampliar_dataset_avanzado(df_usuarios, 200, seed=7).equals(
    ampliar_dataset_avanzado(df_usuarios, 200, seed=8)
)
originales_intactos = df_grande_avanzado.head(100).reset_index(drop=True).equals(
    df_usuarios.reset_index(drop=True)
)
usuarios_nuevos = df_grande_avanzado.iloc[len(df_usuarios):]
fechas_validas = bool(usuarios_nuevos['fecha_registro'].between('2018-01-01', '2024-06-01').all())
print(len(df_grande_avanzado), es_reproducible_avanzado, semillas_distintas, originales_intactos, fechas_validas)


In [ ]:
# ─── Ejercicio A.3 — Validación estadística de los datos sintéticos ─────────
# Un dataset sintético solo es útil si se parece al real. Vas a comprobarlo.
# Trabaja con 'df_usuarios' (originales) y con los 4900 usuarios nuevos de
# 'df_grande_avanzado'.
#
# 1. Guarda en 'porcentaje_tipos_original' y en 'porcentaje_tipos_sintetico'
#    el porcentaje de cada 'tipo_cuenta' (con 1 decimal) entre los usuarios
#    originales y entre los nuevos, respectivamente.
# 2. Guarda en 'media_seguidores_por_tipo' una tabla con una fila por
#    'tipo_cuenta' y dos columnas, 'original' y 'sintetico', con la media de
#    seguidores de cada tipo en cada conjunto.
#    Por ejemplo (esto NO es la solución):
#              original  sintetico
#      free        2758       2530
#      premium     1946       2510
# 3. Guarda en 'fig_comparacion' una figura interactiva de Plotly con
#    diagramas de cajas de 'seguidores' por 'tipo_cuenta', con un color
#    distinto para los usuarios originales y para los sintéticos, y
#    expórtala a '../output_graficos/comparacion_original_sintetico.html'.
# 4. Escribe en un comentario si los datos sintéticos son representativos de
#    los reales y en qué se diferencian (fíjate en las medias y en el
#    máximo de seguidores de los creadores).

usuarios_nuevos = df_grande_avanzado.iloc[len(df_usuarios):]

porcentaje_tipos_original = (df_usuarios['tipo_cuenta'].value_counts(normalize=True) * 100).round(1)
porcentaje_tipos_sintetico = (usuarios_nuevos['tipo_cuenta'].value_counts(normalize=True) * 100).round(1)

media_seguidores_por_tipo = pd.DataFrame({
    'original': df_usuarios.groupby('tipo_cuenta')['seguidores'].mean(),
    'sintetico': usuarios_nuevos.groupby('tipo_cuenta')['seguidores'].mean(),
})

comparacion = pd.concat([
    df_usuarios.assign(origen='original'),
    usuarios_nuevos.assign(origen='sintético'),
])
fig_comparacion = px.box(
    comparacion,
    x='tipo_cuenta',
    y='seguidores',
    color='origen',
    title='Seguidores por tipo de cuenta: usuarios originales frente a sintéticos',
    labels={'tipo_cuenta': 'Tipo de cuenta', 'seguidores': 'Seguidores', 'origen': 'Origen'},
)
fig_comparacion.write_html('../output_graficos/comparacion_original_sintetico.html')
print(porcentaje_tipos_original.to_dict(), porcentaje_tipos_sintetico.to_dict())
print(media_seguidores_por_tipo.round(0))

# Tu razonamiento:
# Los datos sintéticos son representativos en lo esencial: las proporciones de tipos
# de cuenta se parecen (alrededor de 60 / 30 / 10) y las medias de seguidores de cada
# tipo son del mismo orden, sobre todo en los creadores (unos 26.800 en los reales y unos 25.000 en los sintéticos). La
# diferencia está en la forma de la distribución: en los datos sintéticos los
# seguidores son uniformes dentro de un rango, mientras que en los reales hay más
# usuarios concentrados en valores bajos y unos pocos extremos. Además, el máximo de
# los creadores reales (48.138) es una observación concreta, mientras que el
# sintético llega hasta el límite fijado (50.000) con muchos más casos.


In [ ]:
# ─── Ejercicio A.4 — Plotly: barras y línea con eje secundario ──────────────
# Cuando dos magnitudes tienen escalas muy distintas se pueden mostrar en la
# misma figura con dos ejes verticales: uno a cada lado.
#
# 1. Guarda en 'resumen_mensual' una tabla con una fila por mes ('mes',
#    formato 'AAAA-MM') de 'df_procesado', ordenada cronológicamente, con
#    las columnas 'publicaciones' (número de publicaciones del mes) y
#    'engagement_medio' (media del 'engagement_rate' de esas publicaciones).
#    Por ejemplo, si en 2023-05 hubo 3 publicaciones con engagement 0.2, 0.4
#    y 0.6, la fila sería: mes 2023-05, publicaciones 3, engagement_medio
#    0.4 (esto NO es la solución).
# 2. Guarda en 'fig_eje_secundario' una figura de Plotly con barras de las
#    publicaciones por mes (eje vertical izquierdo) y una línea con
#    marcadores del engagement medio (eje vertical derecho). Requisitos:
#    título general, título de los tres ejes, leyenda en la parte inferior y
#    una anotación sobre el mes con más publicaciones que indique su nombre.
# 3. Guarda en 'mes_mas_publicaciones' el mes con más publicaciones.
# 4. Exporta la figura a '../output_graficos/publicaciones_engagement.html'.

resumen_mensual = (
    df_procesado.assign(mes=df_procesado['fecha'].str[:7])
    .groupby('mes')
    .agg(publicaciones=('id_publicacion', 'count'), engagement_medio=('engagement_rate', 'mean'))
    .reset_index()
    .sort_values('mes')
)
mes_mas_publicaciones = resumen_mensual.loc[resumen_mensual['publicaciones'].idxmax(), 'mes']

fig_eje_secundario = make_subplots(specs=[[{'secondary_y': True}]])
fig_eje_secundario.add_trace(
    go.Bar(x=resumen_mensual['mes'], y=resumen_mensual['publicaciones'], name='Publicaciones'),
    secondary_y=False,
)
fig_eje_secundario.add_trace(
    go.Scatter(x=resumen_mensual['mes'], y=resumen_mensual['engagement_medio'],
               mode='lines+markers', name='Engagement medio'),
    secondary_y=True,
)
fig_eje_secundario.add_annotation(
    x=mes_mas_publicaciones,
    y=resumen_mensual['publicaciones'].max(),
    text=f'Más publicaciones: {mes_mas_publicaciones}',
    showarrow=True,
)
fig_eje_secundario.update_layout(
    title_text='Publicaciones y engagement medio por mes',
    legend=dict(orientation='h', y=-0.2),
)
fig_eje_secundario.update_xaxes(title_text='Mes')
fig_eje_secundario.update_yaxes(title_text='Publicaciones', secondary_y=False)
fig_eje_secundario.update_yaxes(title_text='Engagement medio', secondary_y=True)

fig_eje_secundario.write_html('../output_graficos/publicaciones_engagement.html')
print(resumen_mensual.head(3))
print(mes_mas_publicaciones)


In [ ]:
# ─── Ejercicio A.5 — Publicaciones sintéticas con integridad referencial ────
# Al ampliar una tabla hay que ampliar también las que dependen de ella. Vas
# a generar publicaciones nuevas para los usuarios de 'df_grande_avanzado'
# sin romper las relaciones entre tablas.
#
# 1. Fija la semilla 505 (en Faker si lo usas y en el módulo 'random') y
#    guarda en 'df_pubs_nuevas' 1000 publicaciones sintéticas con las mismas
#    columnas y orden que 'df_publicaciones' y estas reglas:
#      - 'id_publicacion': continúa desde el máximo existente + 1
#      - 'id_usuario': elegido entre los ids de 'df_grande_avanzado'
#      - 'fecha': entre la fecha de registro de ESE usuario y el 2024-06-01
#        (texto 'AAAA-MM-DD')
#      - 'tipo_contenido': uno de los que ya existen en 'df_publicaciones'
#      - 'visualizaciones': entre 100 y 50000; 'likes', 'comentarios' y
#        'compartidos' nunca superan las visualizaciones
#      - 'hashtags': de 1 a 3 hashtags, distintos entre sí, elegidos entre
#        los que ya aparecen en 'df_publicaciones' y separados por comas
# 2. Comprueba la coherencia y guarda:
#      'n_sin_usuario'             -> publicaciones cuyo usuario no existe
#      'n_fecha_anterior_registro' -> publicaciones con fecha anterior a la
#                                     de registro de su autor
#      'n_likes_mayores_visualiz'  -> publicaciones con más likes que
#                                     visualizaciones
#      'ids_pub_unicos'            -> si los 'id_publicacion' de todas las
#                                     publicaciones (originales y nuevas) son
#                                     distintos
#    Todos los contadores deben valer 0 y 'ids_pub_unicos' True.
#    Por ejemplo, una publicación del usuario 250 con fecha anterior a su
#    registro contaría como error en 'n_fecha_anterior_registro'.

Faker.seed(505)
random.seed(505)
fake = Faker('es_ES')

id_publicacion_inicial = int(df_publicaciones['id_publicacion'].max()) + 1
ids_usuario = df_grande_avanzado['id_usuario'].tolist()
fecha_registro_por_id = df_grande_avanzado.set_index('id_usuario')['fecha_registro']
tipos_contenido = df_publicaciones['tipo_contenido'].unique().tolist()
hashtags_existentes = sorted({
    hashtag for fila in df_publicaciones['hashtags'].dropna() for hashtag in fila.split(',')
})
fecha_fin = date(2024, 6, 1)

filas = []
for indice in range(1000):
    id_usuario = random.choice(ids_usuario)
    fecha_inicio = date.fromisoformat(fecha_registro_por_id[id_usuario])
    fecha = fake.date_between(fecha_inicio, fecha_fin) if fecha_inicio < fecha_fin else fecha_fin
    visualizaciones = random.randint(100, 50000)
    filas.append({
        'id_publicacion': id_publicacion_inicial + indice,
        'id_usuario': id_usuario,
        'fecha': fecha.isoformat(),
        'tipo_contenido': random.choice(tipos_contenido),
        'likes': float(random.randint(0, visualizaciones)),
        'comentarios': float(random.randint(0, visualizaciones)),
        'compartidos': random.randint(0, visualizaciones),
        'visualizaciones': visualizaciones,
        'hashtags': ','.join(random.sample(hashtags_existentes, random.randint(1, 3))),
    })
df_pubs_nuevas = pd.DataFrame(filas, columns=df_publicaciones.columns).astype(df_publicaciones.dtypes.to_dict())

pubs_con_autor = df_pubs_nuevas.merge(
    df_grande_avanzado[['id_usuario', 'fecha_registro']],
    on='id_usuario', how='left', indicator=True, validate='many_to_one',
)
n_sin_usuario = int((pubs_con_autor['_merge'] != 'both').sum())
n_fecha_anterior_registro = int((pubs_con_autor['fecha'] < pubs_con_autor['fecha_registro']).sum())
n_likes_mayores_visualiz = int((df_pubs_nuevas['likes'] > df_pubs_nuevas['visualizaciones']).sum())
ids_pub_unicos = bool(
    pd.concat([df_publicaciones['id_publicacion'], df_pubs_nuevas['id_publicacion']]).is_unique
)
print(n_sin_usuario, n_fecha_anterior_registro, n_likes_mayores_visualiz, ids_pub_unicos)


### Entrega opcional — `../scripts/pipeline_avanzada.py`

Si quieres consolidar los ejercicios avanzados como funciones reutilizables, implementa en `../scripts/pipeline_avanzada.py`:

| Función | Descripción |
|---|---|
| `exportar_figuras_avanzadas(figuras, carpeta)` | Guarda cada figura de Plotly de un diccionario como `<carpeta>/<nombre>.html`, crea la carpeta si no existe y devuelve la lista de rutas (ejercicio A.1) |
| `ampliar_dataset_avanzado(df_usuarios, n, seed=42)` | Devuelve los usuarios originales seguidos de `n` usuarios ficticios de forma reproducible con la misma `seed`, con ids únicos, mismas columnas y tipos (ejercicio A.2) |

Comprueba tu implementación con:

```bash
python -m pytest ../tests/test_bloque5_avanzado.py -v
```